[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C41_Deep_RL_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**：网络（小 MLP）、反向传播、优化器、replay buffer、**toy 环境** 全部从零手写。

这个 notebook 做四件事：① 确认环境；② 复习 C13 的 MDP 接口、立一个 **GridWorld toy 环境**（全课复用）；③ 用一个**线性 Q + 自举**的最小例子，**亲眼看到致命三要素如何发散**；④ 立下本课纪律——**固定 seed + 稳健阈值** 判定「学到了」。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画回报曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 一个全课复用的 toy 环境：GridWorld

本课所有算法都需要一个能交互的环境。我们从零写一个 **GridWorld**：在 `n×n` 网格上，从左上走到右下角（目标），每步 `-0.01`、到目标 `+1`、撞墙留在原地。接口刻意对齐 Gym 风格：`reset()->s`、`step(a)->(s', r, done)`。

> 这是 C13 表格世界的延续，但本课会把状态喂给**神经网络**（先 one-hot，后续模块用连续特征）。

In [ ]:
class GridWorld:
    '''n x n 网格；状态=格子下标(0..n*n-1)；动作 0上1下2左3右；到右下角终止。'''
    def __init__(self, n=5, step_cost=0.01, seed=0):
        self.n = n
        self.nS = n * n
        self.nA = 4
        self.step_cost = step_cost
        self.goal = n * n - 1
        self.rng = np.random.default_rng(seed)
        self.s = 0
    def reset(self):
        self.s = 0
        return self.s
    def _rc(self, s):
        return s // self.n, s % self.n
    def step(self, a):
        r, c = self._rc(self.s)
        if a == 0: r -= 1
        elif a == 1: r += 1
        elif a == 2: c -= 1
        elif a == 3: c += 1
        r = min(max(r, 0), self.n - 1)          # 撞墙：留在边界内
        c = min(max(c, 0), self.n - 1)
        self.s = r * self.n + c
        done = (self.s == self.goal)
        reward = 1.0 if done else -self.step_cost
        return self.s, reward, done
    def onehot(self, s):
        v = np.zeros(self.nS); v[s] = 1.0
        return v

env = GridWorld(n=5, seed=0)
s = env.reset()
print('初始状态:', s, ' 状态数:', env.nS, ' 动作数:', env.nA, ' 目标:', env.goal)
# 手动走两步验证接口
s1, r1, d1 = env.step(1)   # 下
s2, r2, d2 = env.step(3)   # 右
print(f'step(下)->s={s1}, r={r1}, done={d1}')
print(f'step(右)->s={s2}, r={r2}, done={d2}')
assert s1 == 5 and s2 == 6, 'GridWorld 转移应确定'
assert abs(r1 + 0.01) < 1e-9 and not d1
print('✅ GridWorld toy 环境就绪（全课复用）')

## 3 · 一个随机策略的回报基线

任何「学到了」的断言，都要和一个**基线**比。先测随机策略的平均回报——后续 assert 会要求训练后的策略**显著超过它**。

In [ ]:
def run_episode(env, policy_fn, max_steps=100):
    '''policy_fn(s)->a；返回这一回合的总回报与步数。'''
    s = env.reset()
    total, steps = 0.0, 0
    for _ in range(max_steps):
        a = policy_fn(s)
        s, r, done = env.step(a)
        total += r; steps += 1
        if done: break
    return total, steps

rng = np.random.default_rng(0)
def random_policy(s):
    return int(rng.integers(0, env.nA))

returns = [run_episode(env, random_policy)[0] for _ in range(200)]
baseline = float(np.mean(returns))
print(f'随机策略平均回报 = {baseline:.3f} (200 回合)')
assert baseline < 0.5, '随机策略在 5x5 GridWorld 应远低于最优(最优≈0.9+)'
print('✅ 随机基线已测；后续算法要显著超过它')

## 4 · 致命三要素：亲眼看一个线性 Q 发散

**这是本课最重要的热身。** 致命三要素 = 自举 + 离策略 + 函数逼近，三者同时出现可能让值估计**发散到无穷**。

我们复刻 Sutton & Barto 第 11.2 节的经典最小反例（Baird 风味）：两个状态、一个线性逼近器、离策略地做半梯度 TD 更新——权重会**指数爆炸**。这正是 DQN 必须发明 replay + target network 的原因。

In [ ]:
# 最小发散例子（Baird 风味）：一个被更新的状态 s0 自举到 s1，二者共享同一个权重 w。
# 关键 setup：① 函数逼近——两状态共享 w（v(s)=feat(s)*w）；
#            ② 自举——目标用 gamma*v(s1) 这个自己的估计；
#            ③ 离策略——只反复从 s0 更新（不按任何真实策略的状态分布）。
# 特征：s0 的特征=1，s1 的特征=2（s1 的值是 s0 的两倍，但共享 w）。
feat = np.array([1.0, 2.0])       # feat[s0]=1, feat[s1]=2
gamma = 0.99
w = 1.0                           # 单一共享权重（线性 Q）
alpha = 0.1

norms = []
for t in range(60):
    v_s0 = feat[0] * w               # = w
    v_s1 = feat[1] * w               # = 2w  (自举目标用它)
    td_target = 0.0 + gamma * v_s1   # r=0, 自举
    td_error = td_target - v_s0      # = (2*gamma - 1) * w > 0
    w = w + alpha * td_error * feat[0]   # 半梯度：只对 v_s0 求梯度(乘 feat[0])
    norms.append(abs(w))

print(f'|w|: 初始 {norms[0]:.2f} -> 第10步 {norms[9]:.2f} -> 第30步 {norms[29]:.2f} -> 第60步 {norms[-1]:.2e}')
# 每步 w *= (1 + alpha*(2*gamma-1)) = (1 + 0.1*0.98) = 1.098 -> 指数发散
assert norms[-1] > norms[0] * 5, '致命三要素下权重应发散增长'
assert norms[-1] > norms[29] > norms[9], '应持续指数放大(发散)'
print('💥 权重指数发散！自举(用2w做目标)+离策略(只从s0更新)+逼近(共享w) 三者合谋。')
print('   -> 模块01 的 target network 让目标用「冻结的旧 w」算，正是为打断这个正反馈环。')

## 5 · target network 直觉：冻结/慢更目标就稳了

上面发散的根源：目标 `gamma * v(s1)` 用的是**正在被更新的同一个 w**，形成正反馈。

DQN 的解法——**target network**：算目标时用一份**慢半拍的权重** `w_target`。这里用 **soft update（Polyak）**：每步 `w_target ← τ·w + (1-τ)·w_target`，`τ` 很小（如 0.01）。靶子几乎不动，正反馈被掐断，`|w|` 基本保持有界。

In [ ]:
w = 1.0
w_target = 1.0                    # 慢半拍副本（标量）
alpha = 0.1; tau = 0.01           # 软更新系数
norms_tn = []
for t in range(60):
    v_s0 = feat[0] * w
    v_s1 = feat[1] * w_target        # 用慢半拍权重算目标！
    td_target = 0.0 + gamma * v_s1
    td_error = td_target - v_s0
    w = w + alpha * td_error * feat[0]
    w_target = tau * w + (1 - tau) * w_target   # Polyak 软更新
    norms_tn.append(abs(w))

print(f'带 target net(软更新) 的 |w|: 初始 {norms_tn[0]:.2f} -> 第60步 {norms_tn[-1]:.3f}')
print(f'对比：不带 target net 第60步 |w| = {norms[-1]:.2e}  (爆炸)')
assert norms_tn[-1] < norms[-1] / 50, 'target network 应把发散压低一两个数量级'
assert norms_tn[-1] < 10, '软更新 target net 后 |w| 应基本有界'
print('✅ 慢半拍目标后权重几乎不爆炸（', f'{norms[-1]/norms_tn[-1]:.0f}x 差距', '）—— 模块01两大稳定化技巧之一的核心直觉')

## 6 · 立纪律：固定 seed + 稳健阈值

深度 RL 极度受随机种子摆布（Henderson 2018）。本课每个训练实验都**固定 seed**，并用**稳健阈值**判定成功——断言「方向性 + 留足余量」的判据（如 *最终回报 ≥ 基线 + 间隔*、*后期均值 > 前期均值*），而非脆弱的精确值。

In [ ]:
def improved_over_baseline(final_returns, baseline, margin=0.3):
    '''稳健成功判据：训练后回报均值 >= 基线 + margin。'''
    m = float(np.mean(final_returns))
    return m >= baseline + margin, m

def learning_curve_rises(returns, frac=0.2):
    '''稳健上升判据：后 frac 段均值 > 前 frac 段均值。'''
    k = max(1, int(len(returns) * frac))
    return float(np.mean(returns[-k:])) > float(np.mean(returns[:k]))

# 演示：一条「假装在学习」的回报曲线应被判为上升
fake_curve = list(np.linspace(0.0, 0.8, 50) + rng.normal(0, 0.05, 50))
assert learning_curve_rises(fake_curve), '明显上升的曲线应判为上升'
ok, m = improved_over_baseline(fake_curve[-10:], baseline=0.0, margin=0.3)
assert ok, '末段均值应超过基线+margin'
print(f'后段均值={m:.3f} -> 判定为「学到了」 ✅')
print('这两个判据会贯穿全课所有训练 assert —— 既抓得住真信号，又不被随机波动假阴性。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：每个算法都在自写 toy 环境上、固定 seed 下、用稳健阈值验证「确实学到了」；结构正确则可迁移到 PyTorch + 真实环境。

**你已经看到了全课的灵魂冲突**：函数逼近 + 自举 + 离策略 → 发散；而 target network（模块01）等一系列技巧就是来驯服它的。

**接下来六个模块**：01 DQN → 02 PPO/SAC → 03 离线 RL → 04 世界模型 → 05 探索与前沿。

下一站：**模块 01 · 值函数逼近与 DQN**。